# Automated VWAP Execution Research

This notebook is a separate execution-research layer for the VWAP Probability Band Engine.

It does **not** change the existing VWAP model, probability logic, visual overlay, live runner, or discretionary MT5 workflow.

The purpose is to research whether a rule-based execution layer can convert selected VWAP continuation setups into simulated trades.

Initial focus:

- continuation-only logic
- S-tier and A-tier setups only
- no B-tier discretionary/maybe trades
- no fresh orange entries
- no fresh red entries
- no reversal trades
- no chop/flat-overlay entries
- fixed Nasdaq point execution:
  - SL = -29 points
  - TP = +58 points
  - BE after +29 points

This notebook is research-only. It does not send orders to MT5.

## Workflow

The notebook will eventually follow this structure:

1. Import existing VWAP engine outputs.
2. Build automation-only features.
3. Classify continuation state.
4. Score trend health and band-shift quality.
5. Detect valid green reclaim / green rejection setups.
6. Simulate trade entry and management.
7. Export a trade log.
8. Review results and rejected setups.

Important principle:

The existing visual discretionary engine remains untouched.  
This notebook only adds an optional execution-research layer on top of the already-computed bands and context.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()

# If running from /notebooks, move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
LIVE_ARTIFACTS_DIR = PROJECT_ROOT / "live_artifacts"

PROJECT_ROOT

WindowsPath('c:/GitHub Projects/VWAP-probability-band-engine')

## Automation Configuration

This config controls only the execution-research layer.

It does not alter:

- VWAP calculation
- sigma-band calculation
- probability calibration
- signal-generation internals
- live MT5 overlay behaviour
- existing discretionary workflow

In [2]:
AUTO_CONFIG = {
    # Notebook mode
    "mode": "fully_automated_research",
    "research_only": True,

    # Scope
    "continuation_only": True,
    "allow_longs": True,
    "allow_shorts": True,
    "allow_reversals": False,
    "allow_b_tier": False,
    "allow_fresh_orange_entries": False,
    "allow_fresh_red_entries": False,

    # Setup tiers
    "s_tier_score": 5,
    "a_tier_score": 4,
    "minimum_tradeable_score": 4,

    # Execution
    "entry_mode": "next_bar_open",
    "one_trade_at_a_time": True,

    # Fixed Nasdaq point model
    # Distances are positive in config.
    # PnL result is signed later: SL = -29, TP = +58, BE = 0.
    "stop_loss_points": 29.0,
    "take_profit_points": 58.0,
    "breakeven_trigger_points": 29.0,

    # Risk controls
    "risk_per_trade_pct": 1.0,
    "max_consecutive_losses": 2,
    "daily_profit_cap_pct": 8.0,

    # Session controls
    "session_timezone": "Europe/London",
    "session_start": "14:00",
    "no_new_trades_after": "19:00",

    # Candle quality filters
    "min_body_ratio": 0.25,
    "min_close_through_green": 1.0,
    "max_extension_from_green": 8.0,

    # Shift / trend context
    "shift_lookback": 3,
    "vwap_cross_lookback": 8,
    "max_vwap_crosses_before_chop": 2,
}

AUTO_CONFIG

{'mode': 'fully_automated_research',
 'research_only': True,
 'continuation_only': True,
 'allow_longs': True,
 'allow_shorts': True,
 'allow_reversals': False,
 'allow_b_tier': False,
 'allow_fresh_orange_entries': False,
 'allow_fresh_red_entries': False,
 's_tier_score': 5,
 'a_tier_score': 4,
 'minimum_tradeable_score': 4,
 'entry_mode': 'next_bar_open',
 'one_trade_at_a_time': True,
 'stop_loss_points': 29.0,
 'take_profit_points': 58.0,
 'breakeven_trigger_points': 29.0,
 'risk_per_trade_pct': 1.0,
 'max_consecutive_losses': 2,
 'daily_profit_cap_pct': 8.0,
 'session_timezone': 'Europe/London',
 'session_start': '14:00',
 'no_new_trades_after': '19:00',
 'min_body_ratio': 0.25,
 'min_close_through_green': 1.0,
 'max_extension_from_green': 8.0,
 'shift_lookback': 3,
 'vwap_cross_lookback': 8,
 'max_vwap_crosses_before_chop': 2}

## Execution Sign Convention

The config stores point distances as positive numbers.

For trade PnL:

- stop loss = `-29`
- breakeven = `0`
- take profit = `+58`

For price levels:

Long trade:

- stop = entry - 29
- breakeven trigger = entry + 29
- target = entry + 58

Short trade:

- stop = entry + 29
- breakeven trigger = entry - 29
- target = entry - 58

In [3]:
def build_trade_levels(entry_price: float, side: str, config: dict) -> dict:
    """
    Build fixed-point Nasdaq execution levels.

    Parameters
    ----------
    entry_price:
        Trade entry price.
    side:
        'long' or 'short'.
    config:
        AUTO_CONFIG dictionary.

    Returns
    -------
    dict
        Stop, breakeven trigger, and target prices.
    """
    side = side.lower()

    sl = config["stop_loss_points"]
    tp = config["take_profit_points"]
    be = config["breakeven_trigger_points"]

    if side == "long":
        return {
            "side": "long",
            "entry_price": entry_price,
            "stop_price": entry_price - sl,
            "breakeven_trigger_price": entry_price + be,
            "target_price": entry_price + tp,
        }

    if side == "short":
        return {
            "side": "short",
            "entry_price": entry_price,
            "stop_price": entry_price + sl,
            "breakeven_trigger_price": entry_price - be,
            "target_price": entry_price - tp,
        }

    raise ValueError("side must be 'long' or 'short'")

In [4]:
# Quick sanity check

long_example = build_trade_levels(entry_price=20000.0, side="long", config=AUTO_CONFIG)
short_example = build_trade_levels(entry_price=20000.0, side="short", config=AUTO_CONFIG)

pd.DataFrame([long_example, short_example])

,side,entry_price,stop_price,breakeven_trigger_price,target_price
0,long,20000.0,19971.0,20029.0,20058.0
1,short,20000.0,20029.0,19971.0,19942.0
